###

Run via tmux 
- generate new group-Average CMs (from tryNoHalo/36PscrubBPFD104....) [copy code to ipython]
- fit gradients to them [copy code to ipython]
- fit PFMs to them (.py script here)

In [ ]:
import pandas as pd
import os.path as op
import numpy as np  
import glob
import os

bids_folder=  '/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-dnumrisk'
group_list =  pd.read_csv(op.join(bids_folder, 'group_mapping.csv'), header=0, index_col=0)

subList = group_list.index.tolist()

In [6]:
source_folder = op.join(bids_folder, 'derivatives', 'correlation_matrices.tryNoHalo')

from numrisk.fmri_analysis.gradients.utils import get_basic_mask
mask, labeling_noParcel = get_basic_mask()
N_vertices = len(np.where(mask==True)[0])

ses = 1
task = 'magjudge'
confspec = '36Pscrub3runFD104'

In [ ]:
## Generate new average group correlation matrices

group = 1
subList_fil = group_list[group_list['group']==group].index.tolist()

av_cm = np.zeros((N_vertices, N_vertices)) # matrix with zeros
for sub in subList_fil:
    try:
        sub_file_pattern = op.join(bids_folder,'derivatives','correlation_matrices.tryNoHalo', f'sub-{sub:02d}_ses-{ses}_task-{task}_confspec-{confspec}-*runs_CM-unfiltered.npy')
        sub_file = glob.glob(sub_file_pattern)[0]
        correlation_matrix = np.load(sub_file)
        av_cm += np.arctan(correlation_matrix) # fisher-Z-transformed
        print(f'subject {sub} added')
    except:
        print(f'subject {sub} failed')
        
av_cm = av_cm/len(subList_fil)
av_cm_transf = np.tan(av_cm) # sanity check: diagonal should be 1 !

sub = 'group'+str(group)
np.save(op.join(source_folder, f'sub-{sub}_ses-{ses}_task-{task}_confspec-{confspec}_CM-unfiltered.npy'), av_cm_transf)

subject 1 failed


In [ ]:
### Fit gradients 
group = 1
sub = 'group'+str(group)
key = f'.{confspec}'
target_dir = op.join(bids_folder, 'derivatives', f'gradients{key}', f'sub-{sub}')
os.makedirs(target_dir, exist_ok=True)

kernel = 'normalized_angle'  # 'cosine', 'normalized_angle', 'gaussian'
ztransf = True
specification = f'kernel-{kernel}_ztransf-{ztransf}'
alignRef = '-tanH'
cm_fn = op.join(source_folder, f'sub-{sub}_ses-{ses}_task-{task}_confspec-{confspec}_CM-unfiltered.npy')
cm = np.load(cm_fn)

if ztransf:
    cm = np.arctanh(cm) # "....normalized the correlation coefficients using Fisher’s z-transformation -  # statistische Methode, die den Pearson-Korrelationskoeffizienten (\(r\)) in eine normalverteilte Variable (\(z^{\prime }\)) umwandel = its inverse hyperbolic tangent (artanh).
    cm[np.isinf(cm)] = 0
    cm[np.isnan(cm)] = 0
    print('Applied Fisher z-transformation to connectivity matrix. & handled infs/nans.')

# reference gradients
ref_grad = op.join(bids_folder, 'derivatives', f'gradients.tryParams.36P','sub-All', f'sub-All_gradients_{specification}_avMethod{alignRef}.npy')
grad_ref = np.load(ref_grad)
grad_ref_fil = grad_ref[:,mask].T  # only use nodes in mask


# Fit gradients
import time
start_time = time.time()
from brainspace.gradient import GradientMaps
from brainspace.utils.parcellation import map_to_labels

n_components = 10
gm = GradientMaps(n_components=n_components, alignment='procrustes', kernel=kernel, approach='dm', random_state=0)
gm.fit(cm, reference=grad_ref_fil)

# save results
np.save(op.join(target_dir,f'sub-{sub}_lambdas_{specification}.npy'), gm.lambdas_) 

gm_= gm.gradients_.T 
grad = [None] * n_components
for i, g in enumerate(gm_): # gm.gradients_.T
    grad[i] = map_to_labels(g, labeling_noParcel, mask=mask, fill=np.nan)
np.save(op.join(target_dir,f'sub-{sub}_gradients_{specification}.npy'), grad) 
gm_ = gm.aligned_.T
grad = [None] * n_components
for i, g in enumerate(gm_): # gm.gradients_.T
    grad[i] = map_to_labels(g, labeling_noParcel, mask=mask, fill=np.nan)
fn = op.join(target_dir,f'sub-{sub}_g-aligned{alignRef}_{specification}.npy')
np.save(fn, grad) 

elapsed_time = time.time() - start_time
print(f'Finished sub - {sub} in {elapsed_time/60:.2f} minutes. \n Saved aligned gradients to {fn}')

## OLD

In [ ]:
from utils import plot_nets_CAcolors
hemi_to_plot = 'R'
ref_name = 'dnumrisk-average'

sub = 'group0'
target_folder = op.join(bids_folder,'derivatives','networks_infomap', f'sub-{sub}')
plot_folder = op.join(bids_folder,'plots_and_ims','networks_infomap')
conn_thresholds = [0.03, 0.04, 0.05, 0.1, 0.2, 0.4] 
conn_thresholds_string = "-".join([str(t) for t in conn_thresholds])

fn_consens_mapping = op.join(target_folder, f'sub-{sub}_threshs-{conn_thresholds_string}_ref_name-{ref_name}precFuncMaps-consensMap.npy')
consensus_labels = np.load(fn_consens_mapping)
modules_fsav5 = np.full(mask.shape[0], np.nan, dtype=float)
modules_fsav5[mask] = consensus_labels

figure = plot_nets_CAcolors(modules_fsav5, hemi_to_plot=hemi_to_plot)
figure.suptitle(f'sub {sub}', y=0.75)
plot_fn = op.join(plot_folder, f'sub-{sub}_hemi-{hemi_to_plot}_ref_name-{ref_name}_precFuncMaps-consensMap.png')
figure.savefig(plot_fn, dpi=300, bbox_inches='tight')
#plt.close(figure)